This notebook demonstrates offline video inference and overlay generation using the MediaPipe pipeline.

It reuses the final MediaPipe models selected in notebook 04 and combines:
- MLP classification,
- rule-based feedback,
- rule-based repetition counting,
- video overlay generation.

Note: in this offline demonstration notebook, posture classification is displayed
using a simplified rule-based label for visualization purposes. The final MLP
model is integrated in the real-time application.

In [1]:
import cv2
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
import mediapipe as mp
from collections import deque

In [2]:
# Project structure
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Input / output video paths
INPUT_VIDEO_PATH = DATA_DIR / "raw" / "videos" / "demo" / "pushup_video.mp4"
OUTPUT_VIDEO_PATH = OUTPUT_DIR / "annotated_pushup.mp4"

print(INPUT_VIDEO_PATH)
print(OUTPUT_VIDEO_PATH)

C:\Users\sebas\Documents\master_ai\05_dne\project\posture_checker\data\raw\videos\demo\pushup_video.mp4
C:\Users\sebas\Documents\master_ai\05_dne\project\posture_checker\outputs\annotated_pushup.mp4


In [3]:
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

POSE_LANDMARK_NAMES = [landmark.name for landmark in mp_pose.PoseLandmark]


def compute_angle_3d(a, b, c):
    a = np.array(a, dtype=float)
    b = np.array(b, dtype=float)
    c = np.array(c, dtype=float)

    ba = a - b
    bc = c - b

    norm_ba = np.linalg.norm(ba)
    norm_bc = np.linalg.norm(bc)

    if norm_ba < 1e-8 or norm_bc < 1e-8:
        return np.nan

    cosine = np.dot(ba, bc) / (norm_ba * norm_bc)
    cosine = np.clip(cosine, -1.0, 1.0)
    return np.degrees(np.arccos(cosine))


def get_landmark_point(landmarks, side, joint):
    idx = getattr(mp_pose.PoseLandmark, f"{side.upper()}_{joint.upper()}").value
    lm = landmarks[idx]
    return [lm.x, lm.y, lm.z]


def get_landmark_visibility(landmarks, side, joints):
    vals = []
    for joint in joints:
        idx = getattr(mp_pose.PoseLandmark, f"{side.upper()}_{joint.upper()}").value
        vals.append(landmarks[idx].visibility)
    return float(np.mean(vals))


def choose_body_side_from_landmarks(landmarks):
    key_joints = ["shoulder", "elbow", "wrist", "hip", "knee", "ankle"]
    left_score = get_landmark_visibility(landmarks, "left", key_joints)
    right_score = get_landmark_visibility(landmarks, "right", key_joints)
    return "left" if left_score >= right_score else "right"


def estimate_rep_phase(elbow_angle):
    if np.isnan(elbow_angle):
        return "unknown"
    if elbow_angle < 90:
        return "bottom"
    elif elbow_angle > 140:
        return "top"
    return "transition"

In [4]:
def extract_frame_features_from_landmarks(landmarks):
    side = choose_body_side_from_landmarks(landmarks)

    shoulder = get_landmark_point(landmarks, side, "shoulder")
    elbow = get_landmark_point(landmarks, side, "elbow")
    wrist = get_landmark_point(landmarks, side, "wrist")
    hip = get_landmark_point(landmarks, side, "hip")
    knee = get_landmark_point(landmarks, side, "knee")
    ankle = get_landmark_point(landmarks, side, "ankle")

    elbow_angle_3d = compute_angle_3d(shoulder, elbow, wrist)
    body_alignment_angle_3d = compute_angle_3d(shoulder, hip, ankle)
    hip_angle_3d = compute_angle_3d(shoulder, hip, knee)
    knee_angle_3d = compute_angle_3d(hip, knee, ankle)

    return {
        "selected_side": side,
        "elbow_angle_3d_smooth": elbow_angle_3d,
        "body_alignment_angle_3d_smooth": body_alignment_angle_3d,
        "hip_angle_3d_smooth": hip_angle_3d,
        "knee_angle_3d_smooth": knee_angle_3d,
    }

In [5]:
# Rule-based feedback computed at frame level (real-time)
def generate_frame_feedback(features):
    elbow = features["elbow_angle_3d_smooth"]
    body = features["body_alignment_angle_3d_smooth"]
    hip = features["hip_angle_3d_smooth"]
    phase = estimate_rep_phase(elbow)

    if np.isnan(elbow) or np.isnan(body) or np.isnan(hip):
        return "Pose unclear"

    messages = []

    if body < 150:
        messages.append("Straighten your body")

    if hip < 150:
        messages.append("Lower your hips")

    if body < 135:
        messages.append("Do not sag")

    if elbow > 110 and phase in ["bottom", "transition"]:
        messages.append("Go lower")

    if not messages:
        return "Good posture"

    return " | ".join(messages[:2])

In [6]:
class RepCounter:
    def __init__(self, low_threshold=95, high_threshold=145):
        self.low_threshold = low_threshold
        self.high_threshold = high_threshold
        self.state = "top"
        self.reps = 0

    def update(self, elbow_angle):
        if np.isnan(elbow_angle):
            return self.reps, self.state

        if self.state == "top" and elbow_angle < self.low_threshold:
            self.state = "bottom"

        elif self.state == "bottom" and elbow_angle > self.high_threshold:
            self.reps += 1
            self.state = "top"

        return self.reps, self.state

In [7]:
# Simple rolling smoothing of angle signal (used for rep counting and ML)
angle_buffer = deque(maxlen=7)

def smooth_angle(value):
    angle_buffer.append(value)
    vals = [v for v in angle_buffer if not np.isnan(v)]
    if len(vals) == 0:
        return np.nan
    return float(np.mean(vals))

In [8]:
# Simplified rule-based classification (used for comparison or overlay)
def classify_frame_posture(features):
    elbow = features["elbow_angle_3d_smooth"]
    body = features["body_alignment_angle_3d_smooth"]
    hip = features["hip_angle_3d_smooth"]
    phase = estimate_rep_phase(elbow)

    if np.isnan(elbow) or np.isnan(body) or np.isnan(hip):
        return "Unknown"

    if body < 150 or hip < 150 or (elbow > 110 and phase in ["bottom", "transition"]):
        return "Incorrect"
    return "Correct"

In [9]:
cap = cv2.VideoCapture(str(INPUT_VIDEO_PATH))

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(str(OUTPUT_VIDEO_PATH), fourcc, fps, (width, height))

rep_counter = RepCounter(low_threshold=95, high_threshold=145)

with mp_pose.Pose(
    static_image_mode=False,
    model_complexity=1,
    smooth_landmarks=True,
    enable_segmentation=False,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as pose:

    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(frame_rgb)

        posture_text = "Unknown"
        feedback_text = "Pose unclear"
        rep_phase = "unknown"
        elbow_angle = np.nan
        reps = rep_counter.reps

        if results.pose_landmarks:
            mp_drawing.draw_landmarks(
                frame,
                results.pose_landmarks,
                mp_pose.POSE_CONNECTIONS
            )

            landmarks = results.pose_landmarks.landmark
            features = extract_frame_features_from_landmarks(landmarks)

            elbow_angle_raw = features["elbow_angle_3d_smooth"]
            elbow_angle = smooth_angle(elbow_angle_raw)
            features["elbow_angle_3d_smooth"] = elbow_angle

            rep_phase = estimate_rep_phase(elbow_angle)
            posture_text = classify_frame_posture(features)
            feedback_text = generate_frame_feedback(features)
            reps, _ = rep_counter.update(elbow_angle)

        cv2.putText(frame, f"Posture: {posture_text}", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0) if posture_text == "Correct" else (0, 0, 255), 2)

        cv2.putText(frame, f"Reps: {reps}", (20, 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)

        cv2.putText(frame, f"Phase: {rep_phase}", (20, 120),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

        cv2.putText(frame, f"Elbow angle: {0 if np.isnan(elbow_angle) else int(elbow_angle)}", (20, 160),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

        cv2.putText(frame, f"Feedback: {feedback_text}", (20, height - 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 255), 2)

        writer.write(frame)
        frame_idx += 1

cap.release()
writer.release()

print("Saved annotated video to:", OUTPUT_VIDEO_PATH)

Saved annotated video to: C:\Users\sebas\Documents\master_ai\05_dne\project\posture_checker\outputs\annotated_pushup.mp4
